# Faster R-CNN Coral Detection Training

This notebook trains a Faster R-CNN model on the augmented coral detection dataset from Roboflow.

**Expected Training Time:** ~2-3 hours on T4 GPU (CPU: ~8-10 hours)

**Model:** Faster R-CNN with ResNet50 backbone
- Slower inference (~30-50ms) but highest accuracy
- Better for detailed coral classification

## Step 1: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install roboflow opencv-python matplotlib pandas numpy pillow
print("✓ Dependencies installed successfully")

## Step 2: Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="zdPSV7Poje2dGgQVyLi8")
project = rf.workspace("christiandominiques-workspace").project("coral-detection-e21y1")
dataset = project.versions(1).download("yolov8", location="coral_dataset")

print(f"✓ Dataset downloaded to: {dataset.location}")

## Step 3: Verify Dataset Structure

In [ ]:
import os
import yaml
from pathlib import Path

# Get the dataset path from Roboflow or use default location
if 'dataset' in locals():
    dataset_path = Path(dataset.location)
else:
    # Roboflow may have downloaded to different location
    # Try common locations
    possible_paths = [
        Path("coral_dataset"),
        Path("/content/coral_dataset"),
        Path("datasets/coral-detection-e21y1-1"),
    ]
    
    dataset_path = None
    for path in possible_paths:
        if (path / "data.yaml").exists():
            dataset_path = path
            break
    
    if dataset_path is None:
        # If still not found, list current directory
        print("Looking for dataset in current directory...")
        for item in os.listdir("."):
            if os.path.isdir(item) and (Path(item) / "data.yaml").exists():
                dataset_path = Path(item)
                break
    
    if dataset_path is None:
        raise FileNotFoundError("Could not find dataset with data.yaml. Please check Step 2 output.")

print(f"Using dataset at: {dataset_path}\n")

# Check structure
print("Dataset directory structure:")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(str(dataset_path), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:  # Show first 3 files
            print(f"{subindent}{file}")

# Load and verify data.yaml
data_yaml_path = dataset_path / "data.yaml"
if not data_yaml_path.exists():
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}")

with open(data_yaml_path) as f:
    data = yaml.safe_load(f)

print(f"\n✓ Classes: {data['names']}")
print(f"✓ Number of classes: {len(data['names'])}")

# Count images
try:
    train_count = len(list((dataset_path / "images" / "train").glob("*.jpg")))
    valid_count = len(list((dataset_path / "images" / "val").glob("*.jpg")))
    print(f"✓ Training images: {train_count}")
    print(f"✓ Validation images: {valid_count}")
except Exception as e:
    print(f"Note: Could not count images - {e}")

## Step 4: Prepare Data for Faster R-CNN

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.utils.data as data_utils
from PIL import Image
import xml.etree.ElementTree as ET

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert YOLO format to Faster R-CNN format
def convert_yolo_to_fasterrcnn(yolo_bbox, img_width, img_height):
    """
    Convert YOLO format (center_x, center_y, width, height) normalized [0-1]
    to Faster R-CNN format (x_min, y_min, x_max, y_max) in pixels
    
    Faster R-CNN requires [x_min, y_min, x_max, y_max], NOT [x, y, w, h]
    """
    center_x, center_y, width, height = yolo_bbox
    
    # Check if values are normalized (between 0 and 1) or already in pixels
    if center_x <= 1.0 and center_y <= 1.0 and width <= 1.0 and height <= 1.0:
        # Values are normalized - convert to pixels
        center_x *= img_width
        center_y *= img_height
        width *= img_width
        height *= img_height
    # else: values are already in pixels, use as-is
    
    # Calculate corner coordinates
    x_min = center_x - width / 2.0
    y_min = center_y - height / 2.0
    x_max = center_x + width / 2.0
    y_max = center_y + height / 2.0
    
    # Clamp to image boundaries
    x_min = max(1, min(x_min, img_width - 1))
    y_min = max(1, min(y_min, img_height - 1))
    x_max = max(1, min(x_max, img_width - 1))
    y_max = max(1, min(y_max, img_height - 1))
    
    # Ensure x_min < x_max and y_min < y_max
    if x_min >= x_max:
        x_min, x_max = min(x_min, x_max), max(x_min, x_max)
    if y_min >= y_max:
        y_min, y_max = min(y_min, y_max), max(y_min, y_max)
    
    return [x_min, y_min, x_max, y_max]

class CoralDataset(data_utils.Dataset):
    def __init__(self, image_dir, label_dir, class_names, debug=False):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.class_names = class_names
        self.image_files = sorted(list(self.image_dir.glob("*.jpg")))
        self.debug = debug
        self.debug_count = 0
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        img = Image.open(img_path).convert('RGB')
        img_tensor = torchvision.transforms.ToTensor()(img)
        
        # Load annotations
        label_path = self.label_dir / f"{img_path.stem}.txt"
        
        boxes = []
        labels = []
        
        if label_path.exists():
            with open(label_path) as f:
                for line_num, line in enumerate(f):
                    parts = line.strip().split()
                    if len(parts) < 5:  # Skip invalid lines
                        continue
                    
                    try:
                        class_id = int(parts[0])
                        bbox = [float(x) for x in parts[1:5]]
                        
                        # Debug first few boxes
                        if self.debug and self.debug_count < 3:
                            print(f"  Raw bbox from file: {bbox}")
                            print(f"  Image size: {img.width}x{img.height}")
                        
                        # Convert to Faster R-CNN format: [x_min, y_min, x_max, y_max]
                        rcnn_bbox = convert_yolo_to_fasterrcnn(bbox, img.width, img.height)
                        
                        if self.debug and self.debug_count < 3:
                            print(f"  Converted bbox: {rcnn_bbox}")
                        
                        # Validate bbox
                        x_min, y_min, x_max, y_max = rcnn_bbox
                        if x_min >= x_max or y_min >= y_max:
                            # Skip invalid boxes
                            if self.debug:
                                print(f"  Skipped invalid box: width={x_max-x_min}, height={y_max-y_min}")
                            continue
                        
                        if x_max - x_min < 2 or y_max - y_min < 2:
                            # Skip boxes that are too small
                            continue
                        
                        boxes.append(rcnn_bbox)
                        labels.append(class_id + 1)  # Faster R-CNN uses 1-indexed classes
                        
                        if self.debug and self.debug_count < 3:
                            self.debug_count += 1
                    
                    except (ValueError, IndexError) as e:
                        # Skip lines with parsing errors
                        continue
        
        # If no valid boxes, add a small fallback box
        if not boxes:
            boxes = [[5, 5, 15, 15]]  # Small box in corner
            labels = [1]  # Dummy label
        
        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64)
        }
        
        return img_tensor, target

print("✓ Data loaders prepared for Faster R-CNN")
print("✓ Bounding boxes: Handles both normalized and pixel coordinates")
print("✓ Format output: [x_min, y_min, x_max, y_max] in pixels\n")

# Debug: Check first image
print("Testing dataset with first image...")
test_dataset = CoralDataset(
    train_image_path if 'train_image_path' in locals() else dataset_path / "images" / "train",
    train_label_path if 'train_label_path' in locals() else dataset_path / "labels" / "train",
    data['names'] if 'data' in locals() else [],
    debug=True
)
if len(test_dataset) > 0:
    img, target = test_dataset[0]
    print(f"✓ Sample boxes: {target['boxes']}")
    print(f"✓ Sample labels: {target['labels']}")

## Step 5: Train Faster R-CNN Model

In [ ]:
import torch.optim as optim
from tqdm import tqdm
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

# Load pretrained Faster R-CNN model with updated API
print("Loading pretrained Faster R-CNN model...")
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)

# Get number of input features for the classifier
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Replace the pre-trained head with a new one
num_classes = len(data['names']) + 1  # +1 for background class
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(device)

# Find the correct paths for training data
train_image_path = None
train_label_path = None

# Try different possible directory structures
possible_image_paths = [
    dataset_path / "images" / "train",
    dataset_path / "train" / "images",
    dataset_path / "train",
]

possible_label_paths = [
    dataset_path / "labels" / "train",
    dataset_path / "train" / "labels",
]

for img_path in possible_image_paths:
    if img_path.exists() and len(list(img_path.glob("*.jpg"))) > 0:
        train_image_path = img_path
        break

for lbl_path in possible_label_paths:
    if lbl_path.exists() and len(list(lbl_path.glob("*.txt"))) > 0:
        train_label_path = lbl_path
        break

if train_image_path is None:
    print("❌ ERROR: Could not find training images directory!")
    print(f"Checked paths:")
    for p in possible_image_paths:
        print(f"  - {p} (exists: {p.exists()})")
    raise FileNotFoundError("Training images not found. Check dataset structure from Step 3.")

if train_label_path is None:
    print("❌ ERROR: Could not find training labels directory!")
    print(f"Checked paths:")
    for p in possible_label_paths:
        print(f"  - {p} (exists: {p.exists()})")
    raise FileNotFoundError("Training labels not found. Check dataset structure from Step 3.")

print(f"✓ Found training images at: {train_image_path}")
print(f"  Images: {len(list(train_image_path.glob('*.jpg')))}")
print(f"✓ Found training labels at: {train_label_path}")
print(f"  Labels: {len(list(train_label_path.glob('*.txt')))}\n")

# Create datasets and dataloaders
print("Creating training dataset...")
train_dataset = CoralDataset(
    train_image_path,
    train_label_path,
    data['names']
)

print(f"✓ Training dataset created with {len(train_dataset)} images")

train_loader = data_utils.DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)

# Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training loop
num_epochs = 20
print(f"\nTraining Faster R-CNN for {num_epochs} epochs...")
print(f"Batch size: 4, Learning rate: 0.005\n")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    
    lr_scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

print("\n✓ Training completed!")

## Step 6: Evaluate on Validation Set

In [ ]:
from torchvision.ops import nms

# Find the correct paths for validation data
val_image_path = None
val_label_path = None

# Try different possible directory structures
possible_val_image_paths = [
    dataset_path / "images" / "val",
    dataset_path / "val" / "images",
    dataset_path / "valid" / "images",
    dataset_path / "images" / "valid",
    dataset_path / "val",
]

possible_val_label_paths = [
    dataset_path / "labels" / "val",
    dataset_path / "val" / "labels",
    dataset_path / "labels" / "valid",
    dataset_path / "valid" / "labels",
]

for img_path in possible_val_image_paths:
    if img_path.exists() and len(list(img_path.glob("*.jpg"))) > 0:
        val_image_path = img_path
        break

for lbl_path in possible_val_label_paths:
    if lbl_path.exists() and len(list(lbl_path.glob("*.txt"))) > 0:
        val_label_path = lbl_path
        break

if val_image_path is None or val_label_path is None:
    print("⚠ Warning: Could not find validation set. Skipping validation.")
else:
    print(f"✓ Found validation images at: {val_image_path}")
    print(f"  Images: {len(list(val_image_path.glob('*.jpg')))}")
    print(f"✓ Found validation labels at: {val_label_path}")
    print(f"  Labels: {len(list(val_label_path.glob('*.txt')))}\n")
    
    # Create validation dataset
    val_dataset = CoralDataset(
        val_image_path,
        val_label_path,
        data['names']
    )

    val_loader = data_utils.DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        collate_fn=lambda x: tuple(zip(*x))
    )

    # Evaluation
    model.eval()
    total_detections = 0
    correct_detections = 0
    total_gt = 0

    print("Evaluating on validation set...")
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation"):
            images = [img.to(device) for img in images]
            predictions = model(images)
            
            for pred, target in zip(predictions, targets):
                total_detections += len(pred['boxes'])
                total_gt += len(target['boxes'])
                
                # Simple matching: count detections
                if len(pred['boxes']) > 0:
                    correct_detections += min(len(pred['boxes']), len(target['boxes']))

    if total_detections > 0:
        precision = correct_detections / total_detections
        recall = correct_detections / total_gt if total_gt > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        print(f"\nValidation Results:")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print("No detections made during validation.")


## Step 7: Test on Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_boxes(img, boxes, scores, labels, class_names):
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)
    
    for box, score, label in zip(boxes, scores, labels):
        if score > 0.5:
            x, y, w, h = box[0], box[1], box[2] - box[0], box[3] - box[1]
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y, f"{class_names[label-1]}: {score:.2f}", color='red', fontsize=10)
    
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Find test images
test_images_path = None
possible_test_paths = [
    dataset_path / "images" / "test",
    dataset_path / "test" / "images",
    val_image_path,  # Fall back to validation set if no test set
]

for test_path in possible_test_paths:
    if test_path and test_path.exists():
        test_images_list = list(test_path.glob("*.jpg"))
        if test_images_list:
            test_images_path = test_path
            break

if test_images_path is None:
    print("⚠ Warning: No test images found. Skipping test inference.")
else:
    test_images = list(test_images_path.glob("*.jpg"))[:3]  # Show first 3 images

    print(f"\nTest Predictions ({len(test_images)} images):")
    model.eval()
    with torch.no_grad():
        for img_path in test_images:
            img = Image.open(img_path).convert('RGB')
            img_tensor = torchvision.transforms.ToTensor()(img).to(device)
            
            prediction = model([img_tensor])[0]
            
            print(f"\nImage: {img_path.name}")
            print(f"  Detections: {len(prediction['boxes'])}")
            if len(prediction['scores']) > 0:
                print(f"  Avg Confidence: {prediction['scores'].mean():.4f}")
            
            # Draw predictions
            if len(prediction['boxes']) > 0:
                draw_boxes(img, prediction['boxes'].cpu(), prediction['scores'].cpu(), 
                          prediction['labels'].cpu(), data['names'])

    print("\n✓ Test inference completed!")

## Step 8: Save and Export Model

In [ ]:
import time

# Save the trained model
model_path = "faster_rcnn_coral_best.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': data['names'],
    'num_classes': num_classes
}, model_path)

print(f"✓ Model saved to: {model_path}")
print(f"  File size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

# Measure inference time
model.eval()
test_img = Image.open(test_images[0]).convert('RGB')
test_tensor = torchvision.transforms.ToTensor()(test_img).to(device)

times = []
with torch.no_grad():
    for _ in range(10):
        start = time.time()
        _ = model([test_tensor])
        times.append(time.time() - start)

avg_time = sum(times) / len(times)
print(f"\nInference Time:")
print(f"  Average: {avg_time*1000:.2f} ms")
print(f"  FPS: {1/avg_time:.2f}")

## Step 9: Download Trained Model

In [ ]:
from google.colab import files

# Download the model
print("Downloading trained Faster R-CNN model...")
files.download(model_path)
print(f"✓ Download complete! Model ready for deployment.")